<a href="https://colab.research.google.com/github/reidbarlow/Next_Play_Steam_Games/blob/main/Next_Play_Steam_Games.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Next Play: Steam Games
My app helps streamers decide what game to play and have the best experience by evaluating games with other players playing, what game keeps players playing the longest, and the games with the best reviews.

In [51]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from: /content/drive/MyDrive/


In [64]:
class Game_Finder:
  def __init__(self, games):
    self.games = games

  def online_players(self, n):
    best = self.sort_values("peak_ccu", ascending=False)
    return best[["name", "peak_ccu"]].head(n)

  def average_play_time(self, n):
    longest = self.sort_values("avg_playtime_hours", ascending = False)
    return longest[["name", "avg_playtime_hours"]].head(n)

  def top_positive_games(games, n, min_reviews=90):
    filtered = self[self["num_reviews_total"] >= min_reviews]
    best = filtered.sort_values("pct_pos_total", ascending=False)
    return best[["name", "pct_pos_total"]].head(n)


In [68]:
crowds = online_players(games, 3)
names = list(crowds["name"])
numbers = list(crowds["peak_ccu"])
print("🎮 CROWDS: most players online at once")
print("🥇", names[0], "-", numbers[0], "players")
print("🥈", names[1], "-", numbers[1], "players")
print("🥉", names[2], "-", numbers[2], "players")

stay = average_play_time(games, 3)
names = list(stay["name"])
numbers = list(stay["avg_playtime_hours"])
print("\n⏳ STAYING POWER: longest average play time")
print("🥇", names[0], "-", numbers[0], "hours")
print("🥈", names[1], "-", numbers[1], "hours")
print("🥉", names[2], "-", numbers[2], "hours")

happy = top_positive_games(games, 3)
names = list(happy["name"])
numbers = list(happy["pct_pos_total"])
print("\n⭐ SATISFACTION: highest positive reviews")
print("🥇", names[0], "-", numbers[0], "% positive")
print("🥈", names[1], "-", numbers[1], "% positive")
print("🥉", names[2], "-", numbers[2], "% positive")

🎮 CROWDS: most players online at once
🥇 Counter-Strike 2 - 1212356 players
🥈 Monster Hunter Wilds - 703236 players
🥉 PUBG: BATTLEGROUNDS - 616738 players

⏳ STAYING POWER: longest average play time
🥇 FINAL FANTASY XIV Online - 837.6 hours
🥈 Dota 2 - 717.2 hours
🥉 Counter-Strike 2 - 553.2 hours

⭐ SATISFACTION: highest positive reviews
🥇 HoloCure - Save the Fans! - 99 % positive
🥈 Cats Hidden in Paris - 98 % positive
🥉 VPet-Simulator - 98 % positive


In [73]:
from google import genai
from google.colab import userdata
MODEL = "gemini-flash-lite-latest"

class GameAdvisor:
    def __init__(self):
        # Key comes from Colab Secrets
        self.client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

    def recommend(self, text):
        # Send the text to Gemini and return its answer
        response = self.client.models.generate_content(model=MODEL, contents=text)
        return response.text

message = ("I am a streamer picking a Steam game to play next.\n"
           "Most players online:\n" + str(crowds) + "\n"
           "Longest play time:\n" + str(stay) + "\n"
           "Highest reviews:\n" + str(happy) + "\n"
           "Pick ONE game and explain why in 2 sentences.")

advisor = GameAdvisor()
print("🤖 GEMINI'S PICK")
print(advisor.recommend(message))

🤖 GEMINI'S PICK
Pick **Monster Hunter Wilds**. 

With over 703,000 peak concurrent players, it sits at the absolute sweet spot of massive current hype and high engagement that will draw a huge, active audience to your stream.
